In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

dyslexic_path = '/content/drive/MyDrive/dyslexia_dataset/dyslexic'
control_path = '/content/drive/MyDrive/dyslexia_dataset/control'

print("Dyslexic files:")
print(os.listdir(dyslexic_path))

print("\nControl files:")
print(os.listdir(control_path))

Dyslexic files:
['s11.wav', 's2.wav', 's3(1).wav', 's4(1).m4a', 's6.m4a', 's9.wav', 's2.m4a', 's3.m4a', 's3(1).m4a', 's4(1).wav', 's11.m4a', 's5.m4a', 's3.wav', 's1.m4a', 's8.m4a', 's7.m4a', 's4.wav', 's4.m4a', 's7.wav', 's9.m4a', 's1.wav', 's8.wav', 's12.m4a', 's12.wav', 's6.wav', 's5.wav']

Control files:
['normalstudent1.mp4', 'normalstudent2.mp4', 'normalstudent3.mp4', 'normalstudent2.m4a', 'normalstudent1.m4a', 'normalstudent3.m4a', 'normalstudent2.wav', 'normalstudent1.wav', 'normalstudent3.wav']


In [ ]:
import os

dyslexic_path = '/content/drive/MyDrive/dyslexia_dataset/dyslexic'
control_path = '/content/drive/MyDrive/dyslexia_dataset/control'

# Sirf WAV files lo
dyslexic_files = [f for f in os.listdir(dyslexic_path) if f.endswith('.wav')]
control_files = [f for f in os.listdir(control_path) if f.endswith('.wav')]

print("Dyslexic WAV files:", len(dyslexic_files))
print(dyslexic_files)

print("\nControl WAV files:", len(control_files))
print(control_files)

Dyslexic WAV files: 13
['s11.wav', 's2.wav', 's3(1).wav', 's9.wav', 's4(1).wav', 's3.wav', 's4.wav', 's7.wav', 's1.wav', 's8.wav', 's12.wav', 's6.wav', 's5.wav']

Control WAV files: 3
['normalstudent2.wav', 'normalstudent1.wav', 'normalstudent3.wav']


In [ ]:
import librosa
import numpy as np
import pandas as pd

def extract_features(file_path):
    # Load audio
    y, sr = librosa.load(file_path, sr=16000)

    # Pre-emphasis
    y = librosa.effects.preemphasis(y, coef=0.97)

    # 13 MFCCs extract karo
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13,
                                  n_fft=400, hop_length=160,
                                  n_mels=40)

    # Delta aur Delta-Delta
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    # 39 dimensions = 13 + 13 + 13
    combined = np.vstack([mfcc, delta, delta2])

    # Mean aur Std → 78 dimensions
    mean = np.mean(combined, axis=1)
    std = np.std(combined, axis=1)
    features = np.concatenate([mean, std])

    return features

# Saari files process karo
data = []

for f in dyslexic_files:
    path = os.path.join(dyslexic_path, f)
    features = extract_features(path)
    data.append({'file': f, 'label': 1, **{f'f{i}': v for i, v in enumerate(features)}})

for f in control_files:
    path = os.path.join(control_path, f)
    features = extract_features(path)
    data.append({'file': f, 'label': 0, **{f'f{i}': v for i, v in enumerate(features)}})

df = pd.DataFrame(data)
print("Dataset shape:", df.shape)
print(df[['file', 'label']].to_string())

Dataset shape: (16, 80)
                  file  label
0              s11.wav      1
1               s2.wav      1
2            s3(1).wav      1
3               s9.wav      1
4            s4(1).wav      1
5               s3.wav      1
6               s4.wav      1
7               s7.wav      1
8               s1.wav      1
9               s8.wav      1
10             s12.wav      1
11              s6.wav      1
12              s5.wav      1
13  normalstudent2.wav      0
14  normalstudent1.wav      0
15  normalstudent3.wav      0


In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score

feature_cols = [f'f{i}' for i in range(78)]
X = df[feature_cols].values.astype(float)
y = df['label'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

svm = SVC(kernel='rbf', C=10, gamma=0.001, probability=True)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
    'f1': make_scorer(f1_score, zero_division=0)
}

results = cross_validate(svm, X_scaled, y, cv=cv, scoring=scoring)

print("--- SVM Results ---")
print(f"Accuracy:  {results['test_accuracy'].mean():.4f}")
print(f"Precision: {results['test_precision'].mean():.4f}")
print(f"Recall:    {results['test_recall'].mean():.4f}")
print(f"F1-Score:  {results['test_f1'].mean():.4f}")

--- SVM Results ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

X_train_t = torch.FloatTensor(X_train).unsqueeze(1)
X_test_t  = torch.FloatTensor(X_test).unsqueeze(1)
y_train_t = torch.LongTensor(y_train)
y_test_t  = torch.LongTensor(y_test)

class AudioCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2))
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2))
        self.fc = nn.Sequential(
            nn.Linear(64*19, 256), nn.Dropout(0.4), nn.Linear(256, 2))

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = AudioCNN()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

for epoch in range(50):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/50 - Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    y_pred_cnn = torch.argmax(model(X_test_t), dim=1).numpy()

print("\n--- CNN Results ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_cnn):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_cnn, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_cnn, zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_cnn, zero_division=0):.4f}")

Epoch 10/50 - Loss: 0.0000
Epoch 20/50 - Loss: 0.0000
Epoch 30/50 - Loss: 0.0000
Epoch 40/50 - Loss: 0.0000
Epoch 50/50 - Loss: 0.0000

--- CNN Results ---
Accuracy:  0.7500
Precision: 1.0000
Recall:    0.6667
F1-Score:  0.8000
